<h1>Trabalho de IA</h1>
<br>
<h2>Grupo:</h2>
<ul>
    <li>Diogo Alexandre Alves da Silva - nº 31504</li>
    <li>Gonçalo Rodrigues - nº 31507</li>
    <li>Rui Barbosa - nº 31515</li>
</ul>

<h2>Introdução</h2>
<p>
Este trabalho foi desenvolvido no âmbito da unidade curricular de Inteligência Artificial (2026-27) da licenciatura em Engenharia de Sistemas Informáticos (UPCA). O tema é Comida Local: uma plataforma digital que liga produtores locais, clientes e centros de recolha (armazéns), onde uma frota de veículos recolhe os produtos junto dos produtores e os entrega nos locais escolhidos pelos clientes.
</p>
<h3>Motivação</h3>
<p>
A crescente procura de comida local na UE faz com que os pequenos produtores enfrentem custos logísticos elevados e preços voláteis. Cadeias de abastecimento curtas reduzem emissões de transporte e desperdício alimentar, mas tornam a logística o principal desafio: é preciso escolher rotas eficientes de recolha e entrega que respeitem as janelas de tempo de cada pedido e a capacidade dos veículos.
</p>
<h3>Objetivo</h3>
<p>O objetivo é desenvolver um agente inteligente que planeie essas rotas, aplicando e comparando três tipos de algoritmos:</p>
<ul>
    <li>Pesquisa informada (Greedy e A*)</li>
    <li>Pesquisa local (Hill Climbing)</li>
    <li>Problemas de Satisfação de Restrições (CSP)</li>
</ul>
<p>O trabalho é feito em Python e Jupyter Notebook, sobre um dataset que cobre o distrito de Braga, incluindo a Póvoa de Varzim.</p>
<h3>Estrutura</h3>
<ol>
    <li><b>Introdução:</b> contexto, motivação e objetivo.</li>
    <li><b>Resumo EDA:</b> principais conclusões da análise exploratória (<code>EDA.ipynb</code>).</li>
    <li><b>Problema e abordagem:</b> terminologia, pressupostos, restrições, modelação, função objetivo e algoritmos.</li>
    <li><b>Implementação:</b> código e execução dos algoritmos.</li>
    <li><b>Resultados:</b> melhores rotas, tempos de execução e comparação entre algoritmos.</li>
    <li><b>Conclusão:</b> balanço e melhorias futuras.</li>
</ol>

<h2>Resumo EDA</h2>
<p>
O dataset (versão <i>tiny</i>) descreve 5 produtores, 1 centro de recolha, 2 veículos, 8 clientes, 6 locais de entrega, 10 encomendas com 23 pedidos (cada um com janelas de tempo) e uma matriz com os 132 pares de distâncias e tempos de viagem entre os 12 pontos de rota. A análise completa está em <code>EDA.ipynb</code>.
</p>
<p>
A análise exploratória teve como objetivos: (i) validar a integridade dos dados e a coerência das janelas de tempo (<code>availability_start/end</code> no produtor e <code>delivery_start/end</code> no cliente); (ii) comparar pesos das encomendas com as capacidades dos veículos; (iii) visualizar a distribuição geográfica das paragens. A matriz de distâncias é uma aproximação sintética e simétrica: <code>distance_km</code> é a distância em linha reta multiplicada por um fator de estrada (1.3 por defeito) e <code>duration_min</code> assume uma velocidade média de 45 km/h.
</p>
<p><b>Principais conclusões:</b></p>
<ul>
    <li><b>Integridade:</b> os dados são consistentes, sem valores em falta, duplicados ou referências inválidas entre tabelas.</li>
    <li><b>Janelas de tempo:</b> todas válidas. Considerando o percurso produtor → centro de recolha → cliente, 22 dos 23 pedidos são viáveis. O pedido 8 (encomenda 3) falha por cerca de 28 minutos, pelo que será entregue fora da janela. As janelas de entrega são curtas (1 a 4 horas) e são a restrição mais apertada.</li>
    <li><b>Capacidade:</b> a carga total (~249 kg) é cerca de 10% da capacidade da frota (2454.5 kg) e ~25% do veículo mais pequeno, pelo que a capacidade não é limitativa e o problema é dominado pelas janelas de tempo.</li>
    <li><b>Matriz de distâncias:</b> completa e simétrica, com distância em linha reta × 1.3 e velocidade de 45 km/h. Respeita a desigualdade triangular, pelo que a distância em linha reta é uma heurística admissível para o A*.</li>
    <li><b>Estrutura temporal:</b> os pedidos distribuem-se por 13 dias e formam 8 grupos temporais independentes (o maior com 8 pedidos), que podem ser resolvidos separadamente.</li>
    <li><b>Incoerências a tratar:</b> <code>order_datetime</code> é posterior às janelas na maioria dos pedidos (não é usado como restrição); os veículos indicam ~58-60 km/h contra os 45 km/h da matriz (usa-se a matriz); os produtores 1 e 5 partilham as mesmas coordenadas.</li>
</ul>

<h2>Problema e abordagem</h2>
<h3>Terminologia</h3>
<p>
Uma <b>encomenda</b> (10 no dataset) é composta por um ou mais <b>pedidos</b> (23 no total). Um pedido corresponde a uma linha da tabela <code>order_items</code>: um produto de um produtor, com a respetiva quantidade, janela de recolha e janela de entrega. O pedido é a unidade de planeamento e é representado pela classe <code>Request</code>.
</p>
<h3>Definição do problema</h3>
<p>
Dado um conjunto de pedidos (cada um com local de recolha no produtor, local de entrega no cliente e as respetivas janelas de tempo) e uma frota de veículos com capacidade limitada, determinar as rotas que servem todos os pedidos com o menor custo, cumprindo as restrições.
</p>
<p><b>Pressupostos.</b></p>
<ul>
    <li>Cada veículo parte e termina no centro de recolha a que pertence (<code>center_id</code>).</li>
    <li>O tempo é absoluto (data e hora). Se um veículo chega antes do início de uma janela, espera.</li>
    <li>Os tempos e distâncias de viagem vêm da matriz (45 km/h). Carga e descarga são instantâneas.</li>
    <li>A <code>order_datetime</code> não é usada como restrição (ver EDA).</li>
</ul>
<p><b>Restrições obrigatórias (<i>hard</i>).</b></p>
<ul>
    <li>A recolha ocorre dentro da janela de disponibilidade do produto no produtor, isto é, não antes de <code>availability_start</code> e não depois de <code>availability_end</code>.</li>
    <li>A entrega ocorre a partir do início da janela de entrega do cliente (<code>delivery_start</code>).</li>
    <li>A carga transportada não pode exceder a capacidade do veículo.</li>
    <li>Cada pedido é recolhido e entregue exatamente uma vez.</li>
    <li>Todo o produto passa pelo centro de recolha (armazém): é recolhido no produtor, entregue no armazém e só depois seguido para o cliente.</li>
    <li>Um veículo pode recolher no armazém vários pedidos de uma só vez e entregá-los aos clientes no mesmo percurso, mas só pode sair com produtos que já tenham chegado ao armazém.</li>
</ul>
<p><b>Restrição desejável (<i>soft</i>).</b></p>
<ul>
    <li>A entrega deve ocorrer até ao fim da janela de entrega (<code>delivery_end</code>). Esta restrição pode ser violada com uma penalização, porque o EDA mostra que o pedido 8 não a consegue cumprir, mesmo no melhor caso (atraso mínimo de ~28 min).</li>
</ul>
<h3>Modelação</h3>
<p>
O problema é representado pelas classes <code>Request</code> (pedido), <code>Vehicle</code> (veículo) e <code>State</code> (estado: posição, tempo atual, carga transportada, produtos no armazém e pedidos pendentes/concluídos), com uma função de custo e um verificador de viabilidade de rotas. Cada pedido tem duas fases: produtor → armazém e armazém → cliente.
</p>
completar ao começar a fazer o model.py


<h2>Implementação</h2>
<p><i>[preencher quando o <code>model.py</code> estiver pronto: importar as classes e funções, correr Greedy, A*, Hill Climbing e CSP, e mostrar o resultado de cada um]</i></p>

<h2>Resultados</h2>
<p><i>[preencher: melhores rotas por veículo, tempos de execução e comparação entre algoritmos]</i></p>

<h2>Conclusão</h2>
<p><i>[preencher depois dos resultados: balanço dos resultados, do desenvolvimento e das ferramentas, e melhorias futuras do agente]</i></p>